# 22장 Transformer 직접 구현 — 어텐션을 쌓는다

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `transformer.py` 와 `p01_compare.py` 를 치고 실행. `MultiHeadAttention.forward` 의 shape을 종이에 먼저 | [ch22_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_1_follow.ipynb) |
| 2. 주석 달기 | `forward` 각 줄 뒤에 shape을, `EncoderBlock` 두 잔차 줄에 17장 대응을 주석으로 | [ch22_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `MultiHeadAttention` 복원(가장 어려운 복원 과제). 그다음 14장 패키지에 `--model transformer` 등록 | [ch22_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "22"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/22")

## `transformer.py` — 대상: `MultiHeadAttention.forward`, `EncoderBlock.forward`

In [ ]:
%%writefile transformer.py
# 22장 Transformer 인코더를 직접 만든다: 위치 인코딩, 멀티헤드 어텐션, 인코더 블록, 분류기
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """토큰 순서 정보를 더해 준다. 어텐션 자체는 순서를 모르기 때문."""

    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()                    # [L, 1]
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)                                  # 짝수 차원: sin
        pe[:, 1::2] = torch.cos(pos * div)                                  # 홀수 차원: cos
        self.register_buffer("pe", pe)                                      # 파라미터 아님(학습 안 함), 저장은 됨

    def forward(self, x):                                                   # x: [B, L, d]
        return x + self.pe[: x.size(1)]


class MultiHeadAttention(nn.Module):
    """헤드 h 개가 각자 다른 관점으로 어텐션을 계산하고 결과를 이어 붙인다."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)                                # 질의·키·값을 만드는 선형층
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)                                # 헤드를 합친 뒤 한 번 더

    def forward(self, x, pad_mask=None):
        B, L, d = x.shape
        q = self.wq(x).view(B, L, self.h, self.dk).transpose(1, 2)
        k = self.wk(x).view(B, L, self.h, self.dk).transpose(1, 2)
        v = self.wv(x).view(B, L, self.h, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask[:, None, None, :], float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, L, d)
        return self.wo(out), attn


class EncoderBlock(nn.Module):
    """어텐션 → 잔차+정규화 → 피드포워드 → 잔차+정규화. 17장 ResNet 의 지름길이 여기에도 있다."""

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        a, attn = self.attn(self.norm1(x), pad_mask)
        x = x + self.drop(a)
        x = x + self.drop(self.ff(self.norm2(x)))
        return x, attn


class TransformerClassifier(nn.Module):
    """임베딩 + 위치 → 인코더 블록 n 개 → 패딩 제외 평균 → 분류"""

    def __init__(self, vocab_size, num_classes, d_model=64, n_heads=4, d_ff=128, n_layers=2, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, ids, return_attn=False):                              # ids: [B, L]
        pad_mask = ids == 0
        x = self.pos(self.emb(ids) * math.sqrt(self.emb.embedding_dim))
        attns = []
        for blk in self.blocks:
            x, attn = blk(x, pad_mask)
            attns.append(attn)
        x = self.norm(x)
        keep = (~pad_mask).unsqueeze(-1).float()
        sent = (x * keep).sum(1) / keep.sum(1).clamp(min=1)                 # 19장과 같은 평균 — 다만 문맥이 섞인 뒤
        out = self.fc(sent)
        return (out, attns) if return_attn else out


if __name__ == "__main__":
    torch.manual_seed(0)
    ids = torch.tensor([[5, 3, 9, 0, 0], [7, 2, 4, 8, 1]])
    m = TransformerClassifier(vocab_size=20, num_classes=3)
    out, attns = m(ids, return_attn=True)
    print("파라미터:", f"{sum(p.numel() for p in m.parameters()):,}")
    print("출력:", tuple(out.shape), "| 블록 수", len(attns), "| 어텐션 shape [B, h, L, L]:", tuple(attns[0].shape))
    print("첫 문장 1층 헤드0 어텐션 (패딩 열은 0):")
    for row in attns[0][0, 0].tolist():
        print("  ", [round(v, 2) for v in row])
    pe = PositionalEncoding(8, 6).pe
    print("위치 인코딩 [6, 8] 앞 3행:")
    for row in pe[:3].tolist():
        print("  ", [round(v, 2) for v in row])

### 확인

In [ ]:
!python transformer.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- transformer.py